In [7]:
# RDKIT LIPINSKI RULE OF FIVE 

In [1]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
import os
import csv

# -----------------------------
# Lipinski calculation
# -----------------------------
def lipinski_props(mol):
    if mol is None:
        return None
    
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    h_donors = Lipinski.NumHDonors(mol)
    h_acceptors = Lipinski.NumHAcceptors(mol)
    
    violations = sum([
        mw > 500,
        logp > 5,
        h_donors > 5,
        h_acceptors > 10
    ])
    
    label = "Pass" if violations <= 1 else "Fail"
    
    return [round(mw,2), round(logp,2), h_donors, h_acceptors, violations, label]

# -----------------------------
# Read SMILES
# -----------------------------
def process_smiles_file(file_path):
    data = []
    
    with open(file_path, 'r') as f:
        for line in f:
            smi = line.strip()
            if not smi:
                continue
            
            mol = Chem.MolFromSmiles(smi)
            props = lipinski_props(mol)
            
            data.append([smi] + (props if props else ["Invalid"]*6))
    
    return data

# -----------------------------
# Read SDF
# -----------------------------
def process_sdf_file(file_path):
    data = []
    
    supplier = Chem.SDMolSupplier(file_path)
    
    for mol in supplier:
        if mol is None:
            continue
        
        name = mol.GetProp("_Name") if mol.HasProp("_Name") else "Unknown"
        props = lipinski_props(mol)
        
        data.append([name] + (props if props else ["Invalid"]*6))
    
    return data

# -----------------------------
# Save CSV
# -----------------------------
def save_csv(data, output_path):
    header = ["Molecule", "MW", "LogP", "HDonors", "HAcceptors", "Violations", "Lipinski"]
    
    with open(output_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(data)

# -----------------------------
# Main
# -----------------------------
def main():
    input_path = input("Enter INPUT file path (SMILES or SDF): ").strip()
    output_path = input("Enter OUTPUT CSV file path: ").strip()
    
    if not os.path.exists(input_path):
        print("❌ Input file not found!")
        return
    
    if input_path.endswith(".sdf"):
        data = process_sdf_file(input_path)
    else:
        data = process_smiles_file(input_path)
    
    save_csv(data, output_path)
    
    print(f"\n✅ Results saved to: {output_path}")

# Run
if __name__ == "__main__":
    main()

Enter INPUT file path (SMILES or SDF): C:/Users/ravit/Desktop/TYK2/TESTTYK2_curated.sdf
Enter OUTPUT CSV file path: C:/Users/ravit/Desktop/TYK2/Testlipinskityk2.csv

✅ Results saved to: C:/Users/ravit/Desktop/TYK2/Testlipinskityk2.csv


In [8]:
# RDKIT DESCRIPTORS (~200)

In [3]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import os
import csv

# All RDKit descriptors (name + function pairs)
desc_list = Descriptors.descList
desc_names = [d[0] for d in desc_list]

# -----------------------------
# Compute ALL descriptors
# -----------------------------
def compute_all_descriptors(mol):
    if mol is None:
        return None
    
    values = []
    
    for name, func in desc_list:
        try:
            values.append(func(mol))
        except:
            values.append(None)
    
    return values

# -----------------------------
# SMILES input
# -----------------------------
def process_smiles(file_path):
    data = []
    
    with open(file_path, "r") as f:
        for line in f:
            smi = line.strip()
            if not smi:
                continue
            
            mol = Chem.MolFromSmiles(smi)
            desc = compute_all_descriptors(mol)
            
            data.append([smi] + (desc if desc else ["Invalid"] * len(desc_names)))
    
    return data

# -----------------------------
# SDF input
# -----------------------------
def process_sdf(file_path):
    data = []
    
    supplier = Chem.SDMolSupplier(file_path)
    
    for mol in supplier:
        if mol is None:
            continue
        
        name = mol.GetProp("_Name") if mol.HasProp("_Name") else "Unknown"
        desc = compute_all_descriptors(mol)
        
        data.append([name] + (desc if desc else ["Invalid"] * len(desc_names)))
    
    return data

# -----------------------------
# Save CSV
# -----------------------------
def save_csv(data, output_path):
    header = ["Molecule"] + desc_names
    
    with open(output_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(data)

# -----------------------------
# Main
# -----------------------------
def main():
    input_path = input("Enter INPUT file path (SMILES or SDF): ").strip()
    output_path = input("Enter OUTPUT CSV file path: ").strip()
    
    if not os.path.exists(input_path):
        print("❌ Input file not found!")
        return
    
    if input_path.endswith(".sdf"):
        data = process_sdf(input_path)
    else:
        data = process_smiles(input_path)
    
    save_csv(data, output_path)
    
    print("\n✅ Done!")
    print(f"Descriptors generated: {len(desc_names)}")
    print(f"Saved to: {output_path}")

# Run
if __name__ == "__main__":
    main()

Enter INPUT file path (SMILES or SDF): C:/Users/ravit/Desktop/TYK2/TESTTYK2_curated.sdf
Enter OUTPUT CSV file path: C:/Users/ravit/Desktop/TYK2/Testdescriptors.csv

✅ Done!
Descriptors generated: 208
Saved to: C:/Users/ravit/Desktop/TYK2/Testdescriptors.csv


In [ ]:
# RDKIT FINGEPRINTS GENERATION

In [9]:
from rdkit import Chem
from rdkit.Chem import AllChem, RDKFingerprint
import os
import csv
import numpy as np
from rdkit.DataStructs import ConvertToNumpyArray

# -----------------------------
# Convert fingerprint to array
# -----------------------------
def fp_to_array(fp, size):
    arr = np.zeros((size,), dtype=int)
    ConvertToNumpyArray(fp, arr)
    return arr.tolist()

# -----------------------------
# Generate fingerprints
# -----------------------------
def get_fingerprints(mol):
    if mol is None:
        return None
    
    # Morgan (ECFP) fingerprint
    morgan_fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    morgan_arr = fp_to_array(morgan_fp, 2048)
    
    # RDKit topological fingerprint
    rdk_fp = RDKFingerprint(mol, fpSize=2048)
    rdk_arr = fp_to_array(rdk_fp, 2048)
    
    return morgan_arr + rdk_arr

# -----------------------------
# SMILES input
# -----------------------------
def process_smiles(file_path):
    data = []
    
    with open(file_path, "r") as f:
        for line in f:
            smi = line.strip()
            if not smi:
                continue
            
            mol = Chem.MolFromSmiles(smi)
            fp = get_fingerprints(mol)
            
            if fp:
                data.append([smi] + fp)
    
    return data

# -----------------------------
# SDF input
# -----------------------------
def process_sdf(file_path):
    data = []
    
    supplier = Chem.SDMolSupplier(file_path)
    
    for mol in supplier:
        if mol is None:
            continue
        
        name = mol.GetProp("_Name") if mol.HasProp("_Name") else "Unknown"
        fp = get_fingerprints(mol)
        
        if fp:
            data.append([name] + fp)
    
    return data

# -----------------------------
# Save CSV
# -----------------------------
def save_csv(data, output_path):
    header = ["Molecule"] + \
              [f"Morgan_{i}" for i in range(2048)] + \
              [f"RDKit_{i}" for i in range(2048)]
    
    with open(output_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(data)

# -----------------------------
# Main
# -----------------------------
def main():
    input_path = input("Enter INPUT file path (SMILES or SDF): ").strip()
    output_path = input("Enter OUTPUT CSV file path: ").strip()
    
    if not os.path.exists(input_path):
        print("❌ Input file not found!")
        return
    
    if input_path.endswith(".sdf"):
        data = process_sdf(input_path)
    else:
        data = process_smiles(input_path)
    
    save_csv(data, output_path)
    
    print("\n✅ Fingerprints generated successfully!")
    print(f"Total features per molecule: 4096 (2048 Morgan + 2048 RDKit)")
    print(f"Saved to: {output_path}")

# Run
if __name__ == "__main__":
    main()

Enter INPUT file path (SMILES or SDF): C:/Users/ravit/Desktop/TYK2/TESTTYK2_curated.sdf
Enter OUTPUT CSV file path: C:/Users/ravit/Desktop/TYK2/Testfingerprints.csv

✅ Fingerprints generated successfully!
Total features per molecule: 4096 (2048 Morgan + 2048 RDKit)
Saved to: C:/Users/ravit/Desktop/TYK2/Testfingerprints.csv
